In [9]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# thresholds: min total minutes in N, min total minutes in N+1, min attempts for a shooting %
MIN_N, MIN_NEXT, MIN_ATT = 300, 200, 50

# feature table saved by 02_features
df = pd.read_csv("data/processed/player_seasons_features.csv")

In [10]:
# keep rows with reliable features (season N) and a reliable target (N+1)
labeled = df[(df["min_total"] >= MIN_N) & (df["min_next_total"] >= MIN_NEXT)]

# temporal split by season N (never random)
train = labeled[labeled["season_year"] <= 2020].copy()
val   = labeled[labeled["season_year"].between(2021, 2022)].copy()
test  = labeled[labeled["season_year"] >= 2023].copy()

In [11]:
# split sizes
print(len(train))
print(len(val))
print(len(test))

2647
695
697


In [12]:
# share of train rows with few attempts (used to pick MIN_ATT)
for t in [20, 50, 100]:
    print(t,
          round((train["FG3A_total"] < t).mean(), 3),
          round((train["FTA_total"] < t).mean(), 3))

20 0.175 0.032
50 0.247 0.18
100 0.362 0.452


In [13]:
# shooting % is unreliable with few attempts: set it to NaN (the row is kept)
def clean_pcts(d, min_att=MIN_ATT):
    d = d.copy()
    d.loc[d["FG3A_total"] < min_att, "FG3_PCT"] = np.nan
    d.loc[d["FTA_total"] < min_att, "FT_PCT"] = np.nan
    return d

# same rule on every split
train, val, test = clean_pcts(train), clean_pcts(val), clean_pcts(test)

In [14]:
# raw data: zero total attempts can still show a pct (per-game FG3A is rounded)
df.loc[df["FG3A_total"] == 0, "FG3_PCT"].unique()

array([0.   , 1.   , 0.333, 0.25 , 0.5  ])

In [15]:
# baseline 1: predict no change, so the error is the target itself
mae_1 = val["target"].abs().mean()
rmse_1 = (val["target"] ** 2).mean() ** 0.5

print("baseline 1 (no change)    MAE:", round(mae_1, 3), " RMSE:", round(rmse_1, 3))

# baseline 2: average PPG of the last 3 seasons (NaN lags are skipped)
pred_2 = val[["PTS", "ppg_lag1", "ppg_lag2"]].mean(axis=1)
err_2 = val["ppg_next"] - pred_2

mae_2 = err_2.abs().mean()
rmse_2 = (err_2 ** 2).mean() ** 0.5

print("baseline 2 (3-season avg) MAE:", round(mae_2, 3), " RMSE:", round(rmse_2, 3))

2.356115107913669
3.0408797477519185
2.7513669064748205
3.537036541796197


In [16]:
# mean change by age bucket (train only): is there signal in age?
bins = [18, 22, 25, 28, 31, 34, 45]
print(train.groupby(pd.cut(train["AGE"], bins), observed=True)["target"]
           .agg(["count", "mean", "std"]))

          count      mean       std
AGE                                
(18, 22]    442  1.572172  3.137892
(22, 25]    727  0.400688  3.167307
(25, 28]    678 -0.312094  2.999764
(28, 31]    458 -0.890830  2.882818
(31, 34]    231 -1.422944  2.446744
(34, 45]    111 -1.700901  2.411808
